# Earnalism stills job `stills-film-01-r2`

**Prepared, not run.** Input zip SHA256 `507d1a3213815cc0b34cb99192c95cb7925386253265af27f8f2f0e27c1cd3ee`. Copy `inputs.zip` to `/content/drive/MyDrive/earnalism-audiovisual/bicharak/stills-film-01-r2/` in Drive, set `RUN = True` in the first cell, then run all. Outputs and `manifest.json` (sha256 of every output) land in the same folder.


In [ ]:
RUN = False

In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import pathlib
JOB_ID = 'stills-film-01-r2'
TITLE = 'bicharak'
KIND = 'stills'
INPUT_ZIP_SHA256 = '507d1a3213815cc0b34cb99192c95cb7925386253265af27f8f2f0e27c1cd3ee'
MODEL_REPO = 'black-forest-labs/FLUX.1-schnell'
MODEL_REVISION = 'resolved-at-run-time'
STEPS = 4
DRIVE_JOB_DIR = '/content/drive/MyDrive/earnalism-audiovisual/bicharak/stills-film-01-r2'


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import hashlib, json, pathlib, zipfile
from google.colab import drive
drive.mount("/content/drive")
OUT_DIR = pathlib.Path(DRIVE_JOB_DIR)
INPUT_ZIP = OUT_DIR / "inputs.zip"
if not INPUT_ZIP.is_file():
    raise RuntimeError("Copy inputs.zip into " + str(OUT_DIR) + " first")
actual = hashlib.sha256(INPUT_ZIP.read_bytes()).hexdigest()
if actual != INPUT_ZIP_SHA256:
    raise RuntimeError("inputs.zip sha256 mismatch: " + actual)
with zipfile.ZipFile(INPUT_ZIP) as z:
    JOB = json.loads(z.read("job.json"))
if JOB["job_id"] != JOB_ID:
    raise RuntimeError("job.json belongs to a different job")
print("inputs verified for", JOB_ID)


In [ ]:
# Stills: FLUX.1-schnell, 4 steps. The exact revision is resolved now and recorded.
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers", "transformers",
                "accelerate", "sentencepiece", "protobuf"], check=True)
import torch
from diffusers import FluxPipeline
from huggingface_hub import HfApi

revision = HfApi().model_info(MODEL_REPO).sha
MODEL_INFO = {"repo": MODEL_REPO, "revision": revision, "steps": STEPS}
pipe = FluxPipeline.from_pretrained(MODEL_REPO, revision=revision, torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()
for item in JOB["items"]:
    gen = torch.Generator("cpu").manual_seed(item["seed"])
    image = pipe(item["prompt"], num_inference_steps=STEPS, guidance_scale=0.0,
                 max_sequence_length=256, width=item["width"], height=item["height"],
                 generator=gen).images[0]
    image.save(OUT_DIR / (item["key"] + ".png"))
    print("wrote", item["key"])


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import hashlib, json
outputs = {}
for p in sorted(OUT_DIR.iterdir()):
    if p.is_file() and p.name not in ("manifest.json", "inputs.zip"):
        outputs[p.name] = hashlib.sha256(p.read_bytes()).hexdigest()
if not outputs:
    raise RuntimeError("no outputs were produced")
manifest = {"schema": "earnalism.audiovisual-outputs.v1", "job_id": JOB_ID, "title": TITLE,
            "kind": KIND, "input_zip_sha256": INPUT_ZIP_SHA256, "model": MODEL_INFO,
            "outputs": outputs}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")
print("manifest.json written:", len(outputs), "files")
